[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcivardi/divelab/blob/main/notebooks/03_Buoyancy_Feedback.ipynb)

# DiveLab

## Notebook 03 — Buoyancy, Instability and Positive Feedback

**Guiding question:** Why can a neutrally buoyant diver become unstable after a very small vertical disturbance?

*Small disturbances can be amplified when buoyancy depends on depth.*

## Learning objectives

By the end of this lab, you will be able to:

- relate gas expansion to buoyant force;
- construct a neutrally buoyant equilibrium at a chosen depth;
- explain why that equilibrium can be unstable;
- simulate both accelerated ascent and accelerated descent;
- interpret hydrodynamic drag and its quadratic dependence on velocity;
- visualize the dynamics in the phase plane;
- linearize the model near equilibrium and interpret its eigenvalues.

## From Notebook 02 to Notebook 03

Notebook 02 showed that gas volume increases nonlinearly during ascent and decreases during descent.

Now we add Archimedes' principle:

$$
F_B = \rho g V
$$

If part of the displaced volume is gas-filled, then buoyancy depends on pressure, and therefore on depth.

That creates a feedback mechanism.

## Positive feedback in both directions

Suppose the diver is neutrally buoyant at depth $z_e$.

A small upward disturbance gives:

$$
z \downarrow
\Rightarrow
P \downarrow
\Rightarrow
V_g \uparrow
\Rightarrow
F_B \uparrow
\Rightarrow
a_\text{up} \uparrow
\Rightarrow
z \downarrow
$$

A small downward disturbance gives the mirror image:

$$
z \uparrow
\Rightarrow
P \uparrow
\Rightarrow
V_g \downarrow
\Rightarrow
F_B \downarrow
\Rightarrow
a_\text{down} \uparrow
\Rightarrow
z \uparrow
$$

So the equilibrium can be unstable in both directions.

## The "pallonata"

An uncontrolled accelerating ascent is often called a **pallonata** in diving slang.

In control-systems language, it is a runaway response driven by positive feedback.

The same underlying instability can also amplify a downward disturbance.

So the deeper concept is not merely "runaway ascent", but:

> **an unstable vertical equilibrium**

## Model conventions

We use:

- depth $z>0$ measured downward from the surface;
- vertical velocity $v>0$ measured upward.

Therefore:

$$
\frac{dz}{dt} = -v
$$

A positive $v$ means the diver is ascending and depth is decreasing.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Physical constants

In [ ]:
rho = 1025.0        # seawater density [kg/m^3]
g = 9.80665         # gravitational acceleration [m/s^2]
P0 = 101325.0       # atmospheric pressure [Pa]

## Pressure and Boyle's law

In [ ]:
def pressure_at_depth(depth_m):
    return P0 + rho * g * depth_m


def gas_volume_at_depth(depth_m, surface_volume):
    return surface_volume * P0 / pressure_at_depth(depth_m)

## Buoyancy model

We split the diver's displaced volume into:

- a fixed incompressible part $V_f$;
- a compressible gas part $V_g(z)$.

The buoyant force is:

$$
F_B(z)=\rho g \left(V_f + V_g(z)\right)
$$

## Choose the equilibrium depth

We choose:

$$
z_e=20\ \text{m}
$$

and build the model so that the diver is exactly neutrally buoyant there.

That means:

$$
F_B(z_e) = mg
$$

In [ ]:
mass = 90.0                       # kg
equilibrium_depth = 20.0          # m
gas_surface_volume = 0.005        # m^3 = 5 L

## Compute the required fixed volume

At equilibrium:

$$
\rho g (V_f + V_g(z_e)) = mg
$$

so:

$$
V_f = \frac{m}{\rho} - V_g(z_e)
$$

In [ ]:
gas_volume_eq = gas_volume_at_depth(
    equilibrium_depth,
    gas_surface_volume
)

fixed_volume = mass / rho - gas_volume_eq

print(f"Gas volume at equilibrium: {gas_volume_eq*1000:.3f} L")
print(f"Required fixed volume:     {fixed_volume*1000:.3f} L")

In [ ]:
def buoyant_force(depth_m):
    gas_volume = gas_volume_at_depth(depth_m, gas_surface_volume)
    total_volume = fixed_volume + gas_volume
    return rho * g * total_volume


def weight_force():
    return mass * g

## Verify the equilibrium

In [ ]:
fb_eq = buoyant_force(equilibrium_depth)
fw = weight_force()

print(f"Buoyant force: {fb_eq:.3f} N")
print(f"Weight force:  {fw:.3f} N")
print(f"Net force:     {fb_eq - fw:+.6f} N")

At $z=z_e$ and $v=0$, the net vertical force is zero.

But zero net force does **not** imply stability.

We now test what happens after tiny perturbations.

## Hydrodynamic drag

For a body moving through water, a common model is:

$$
F_D = \frac12 \rho C_D A\,v|v|
$$

where:

- $\rho$ is water density;
- $C_D$ is the drag coefficient;
- $A$ is the frontal area;
- $v$ is vertical velocity.

The term $v|v|$ preserves direction.

For $v>0$ (upward motion), drag acts downward.

For $v<0$ (downward motion), drag acts upward.

Thus drag always opposes motion.

## Why quadratic drag?

At moderate and high Reynolds number, drag is approximately proportional to $v^2$.

The factor

$$
\frac12 \rho C_D A
$$

collects the effect of fluid density, body shape and frontal area.

In this notebook, $C_D$ and $A$ are simplified effective parameters for the diver.

In [ ]:
Cd = 0.9        # effective drag coefficient
A = 0.7         # effective frontal area [m^2]


def drag_force(velocity_m_s):
    return 0.5 * rho * Cd * A * velocity_m_s * abs(velocity_m_s)

## Equation of motion

With upward velocity positive:

$$
m\frac{dv}{dt}
=
F_B(z)-mg-F_D(v)
$$

and:

$$
\frac{dz}{dt}=-v
$$

The complete nonlinear system is therefore:

$$
\dot z = -v
$$

$$
\dot v =
\frac{
F_B(z)-mg-\frac12\rho C_D A v|v|
}{m}
$$

In [ ]:
def acceleration(depth_m, velocity_m_s):
    return (
        buoyant_force(depth_m)
        - weight_force()
        - drag_force(velocity_m_s)
    ) / mass

## Check the restoring or destabilizing force

If the equilibrium were stable, a small upward displacement would create a downward restoring force.

Let's test the sign of the net force around 20 m.

In [ ]:
for d in [19.5, 19.9, 20.0, 20.1, 20.5]:
    net = buoyant_force(d) - weight_force()
    print(f"{d:4.1f} m → net upward force = {net:+.3f} N")

You should observe:

- shallower than 20 m → positive upward force;
- deeper than 20 m → negative upward force.

So a displacement is reinforced instead of corrected.

That is the signature of an unstable equilibrium.

## Numerical simulation

In [ ]:
def simulate(
    depth0,
    velocity0,
    duration=30.0,
    dt=0.01
):
    n = int(duration / dt) + 1
    t = np.linspace(0, duration, n)

    depth = np.zeros(n)
    velocity = np.zeros(n)

    depth[0] = depth0
    velocity[0] = velocity0

    for i in range(n - 1):
        a = acceleration(depth[i], velocity[i])

        velocity[i + 1] = velocity[i] + a * dt
        depth[i + 1] = depth[i] - velocity[i + 1] * dt

        if depth[i + 1] <= 0:
            depth[i + 1:] = 0
            velocity[i + 1:] = velocity[i + 1]
            break

    return t, depth, velocity

## Two symmetric perturbations

We start exactly at the equilibrium depth, but give the diver:

- a tiny upward velocity;
- a tiny downward velocity.

In [ ]:
t_up, z_up, v_up = simulate(
    depth0=equilibrium_depth,
    velocity0=+0.05
)

t_down, z_down, v_down = simulate(
    depth0=equilibrium_depth,
    velocity0=-0.05
)

In [ ]:
plt.plot(t_up, z_up, label="Upward perturbation")
plt.plot(t_down, z_down, label="Downward perturbation")
plt.axhline(equilibrium_depth, linestyle="--")

plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("Divergence from an unstable equilibrium")
plt.grid(True)
plt.legend()
plt.show()

## Interpretation

The two trajectories move away from the same equilibrium:

- one toward shallower water;
- one toward greater depth.

This is a direct visualization of instability.

In [ ]:
plt.plot(t_up, v_up, label="Upward perturbation")
plt.plot(t_down, v_down, label="Downward perturbation")
plt.axhline(0, linestyle="--")

plt.xlabel("Time [s]")
plt.ylabel("Upward velocity [m/s]")
plt.title("Velocity after opposite perturbations")
plt.grid(True)
plt.legend()
plt.show()

## Phase plane

The state of the system is:

$$
x =
\begin{bmatrix}
z \\
v
\end{bmatrix}
$$

The phase plane plots velocity against depth.

The equilibrium is:

$$
(z_e,v_e)=(20,0)
$$

If the equilibrium is unstable, trajectories should move away from this point.

In [ ]:
plt.plot(z_up, v_up, label="Upward perturbation")
plt.plot(z_down, v_down, label="Downward perturbation")
plt.scatter([equilibrium_depth], [0], s=60, label="Equilibrium")

plt.xlabel("Depth z [m]")
plt.ylabel("Upward velocity v [m/s]")
plt.title("Phase portrait near the equilibrium")
plt.grid(True)
plt.legend()
plt.show()

## Local buoyancy sensitivity

The destabilizing mechanism comes from the slope:

$$
\frac{dF_B}{dz}
$$

Since gas volume decreases as depth increases:

$$
\frac{dF_B}{dz}<0
$$

Therefore:

- moving shallower increases buoyancy;
- moving deeper decreases buoyancy.

In [ ]:
def buoyancy_sensitivity(depth_m):
    return (
        -rho * g
        * gas_surface_volume
        * P0
        * rho * g
        / (P0 + rho * g * depth_m) ** 2
    )

k_b = buoyancy_sensitivity(equilibrium_depth)

print(f"dF_B/dz at equilibrium = {k_b:.3f} N/m")

## Linearization near equilibrium

Let:

$$
\delta z = z-z_e
$$

and, since the equilibrium velocity is $v_e=0$:

$$
\delta v = v-v_e = v
$$

Near equilibrium:

$$
F_B(z)-mg
\approx
\left.\frac{dF_B}{dz}\right|_{z_e}\delta z
$$

The quadratic drag behaves like $v|v|$, whose derivative at $v=0$ is zero.

So, to first order, the linearized system is:

$$
\delta\dot z = -\delta v
$$

$$
\delta\dot v =
\frac{k_b}{m}\delta z
$$

with $k_b<0$.

## State-space form

The linearized dynamics are:

$$
\begin{bmatrix}
\delta\dot z \\
\delta\dot v
\end{bmatrix}
=
\begin{bmatrix}
0 & -1 \\
k_b/m & 0
\end{bmatrix}
\begin{bmatrix}
\delta z \\
\delta v
\end{bmatrix}
$$

Because $k_b<0$, the matrix has one positive and one negative real eigenvalue.

That means the equilibrium is a **saddle point**.

In [ ]:
A_lin = np.array([
    [0.0, -1.0],
    [k_b / mass, 0.0]
])

eigvals, eigvecs = np.linalg.eig(A_lin)

print("Linearized A matrix:")
print(A_lin)
print()
print("Eigenvalues:")
print(eigvals)

## Why drag does not appear in the linear model

This is an important subtlety.

Quadratic drag is:

$$
F_D \propto v|v|
$$

and therefore:

$$
\left.
\frac{dF_D}{dv}
\right|_{v=0}
=0
$$

So quadratic drag does not contribute a first-order damping term at the equilibrium.

Farther from equilibrium, however, drag becomes increasingly important and limits the growth of velocity.

## Nonlinear phase portrait

Let's launch several trajectories around the equilibrium.

In [ ]:
initial_conditions = [
    (19.8, 0.0),
    (20.2, 0.0),
    (20.0, 0.03),
    (20.0, -0.03),
    (19.9, 0.03),
    (20.1, -0.03),
]

for z0, v0 in initial_conditions:
    t_i, z_i, v_i = simulate(
        depth0=z0,
        velocity0=v0,
        duration=20.0
    )
    plt.plot(z_i, v_i)

plt.scatter([equilibrium_depth], [0], s=70)

plt.xlabel("Depth z [m]")
plt.ylabel("Upward velocity v [m/s]")
plt.title("Nonlinear phase portrait around the unstable equilibrium")
plt.grid(True)
plt.show()

## Control-systems interpretation

The system contains:

- an equilibrium point;
- a nonlinear state-dependent force;
- positive feedback through compressible gas;
- nonlinear damping through hydrodynamic drag.

The instability is generated by the buoyancy-depth coupling.

Drag limits motion but does not turn this equilibrium into a stable one.

## Why this matters for a "pallonata"

An accelerating ascent can begin with a very small upward disturbance.

Once ascent starts:

> depth decreases → pressure decreases → gas expands → buoyancy increases → ascent accelerates

Near the surface, the relative gas expansion becomes larger, making this mechanism increasingly important.

The same mathematics also explains why a downward disturbance can develop into an accelerating descent if the diver does not correct buoyancy.

## Exercises

### 1. Smaller perturbations

Repeat the simulation with:

```python
velocity0 = +0.01
velocity0 = -0.01
```

Do the trajectories still diverge?

### 2. Start with a depth displacement only

Set:

```python
depth0 = 19.9
velocity0 = 0
```

and then:

```python
depth0 = 20.1
velocity0 = 0
```

Observe what happens.

### 3. Change frontal area

Try:

```python
A = 0.4
A = 0.7
A = 1.0
```

How does drag change the resulting velocities?

Does it change the local linear stability?

### 4. Change gas volume

Try different values of `gas_surface_volume`.

How do the eigenvalues change?

What does that tell you about the strength of the instability?

## Challenge — stable manifold and unstable manifold

The linearized equilibrium is a saddle.

Use the eigenvectors of `A_lin` to plot the two local eigendirections through the equilibrium.

Then compare them with the nonlinear trajectories.

This is our first explicit connection between scuba physics and classical dynamical-systems analysis.

In [ ]:
# Your code here

## Summary

In this lab we learned that:

- neutral buoyancy does not necessarily imply stability;
- gas compression and expansion make buoyancy depend on depth;
- small vertical disturbances can be amplified;
- both accelerated ascent and accelerated descent are possible;
- hydrodynamic drag opposes motion and grows approximately with $v^2$;
- quadratic drag does not provide first-order damping at $v=0$;
- the local linearized equilibrium is a saddle point;
- the phase plane makes the instability visually clear.

### Core insight

> **The diver can be neutrally buoyant and still dynamically unstable.**

### Next

The next notebook can introduce an explicit control input — BCD inflation and venting — and ask:

> How can feedback control stabilize an unstable buoyancy system?